# CP2 Week 1 -- Schema Validation

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** CP1 complete (v1 pipeline working)
**Focus:** validate_schema(), meaningful errors, data contracts

## Learning Objectives
- Understand why schema validation prevents silent failures
- Write a `validate_schema()` function with clear error messages
- Create data contracts that specify expected structure
- Integrate validation into your v2 pipeline
- Handle edge cases gracefully

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Why Validate Schemas?

In CP1, your v1 pipeline assumed data always had the right columns and types. That works when you control the data, but real-world data is messy:

- Someone renames a column from `temp` to `temperature`
- A CSV export adds an extra column you did not expect
- A number field contains the string `"N/A"` instead of a number
- The file is completely empty

Without validation, your pipeline **silently produces wrong results** or **crashes with a confusing error** deep inside your code.

**Schema validation** checks the structure of your data BEFORE processing, catching problems early with clear, actionable messages.

### Pipeline Without vs. With Validation

```
WITHOUT VALIDATION:
  Raw Data --> clean_data() --> CRASH (line 47: KeyError 'temp')
  (You waste 20 minutes debugging)

WITH VALIDATION:
  Raw Data --> validate_schema() --> 'Missing column: temp. Got: temperature'
  (You fix it in 10 seconds)
```

### Why This Matters

Every professional data pipeline validates its inputs. Think of it like a bouncer at a club -- checking IDs before anyone gets in. If bad data sneaks past, every downstream result is suspect. In your v2 pipeline, `validate_schema()` is the first line of defense.

### What Is a Schema?

A **schema** is a description of what your data should look like:

| Property | Example |
|----------|---------|
| Required columns | `['id', 'value', 'status']` |
| Column types | `{'value': 'numeric', 'status': 'string'}` |
| Value ranges | `{'value': (0, 100)}` |
| Non-null columns | `['id', 'value']` |

You store the schema in your config dictionary, so it is easy to update.

### Example 1: Check Required Columns (Simplest Version)

In [ ]:
# Simplest schema validation: just check column names

def validate_columns(data, required_columns):
    """Check that all required columns are present.
    
    Args:
        data: list of dicts (each dict is a row)
        required_columns: list of column names that must exist
    
    Returns:
        True if all columns present
    
    Raises:
        ValueError with helpful message if columns missing
    """
    if not data:
        raise ValueError("Data is empty -- nothing to validate")
    
    actual = set(data[0].keys())
    required = set(required_columns)
    missing = required - actual
    
    if missing:
        raise ValueError(
            "Missing " + str(len(missing)) + " required column(s): "
            + str(sorted(missing)) + "\n"
            "Available columns: " + str(sorted(actual))
        )
    
    print("All " + str(len(required)) + " required columns present.")
    return True

# Test with GOOD data
good_data = [
    {"id": 1, "value": "25.0", "status": "ok"},
    {"id": 2, "value": "30.0", "status": "ok"},
]
validate_columns(good_data, ["id", "value", "status"])
print("Test 1 PASSED")

**Expected Output:**
```
All 3 required columns present.
Test 1 PASSED
```

### Example 2: Catching Missing Columns

In [ ]:
# Test with BAD data -- missing columns
bad_data = [{"id": 1, "temperature": 25.0}]

try:
    validate_columns(bad_data, ["id", "value", "status"])
except ValueError as e:
    print("Caught error:")
    print(e)

**Expected Output:**
```
Caught error:
Missing 2 required column(s): ['status', 'value']
Available columns: ['id', 'temperature']
```

### Example 3: Handling Empty Data

In [ ]:
# Test with EMPTY data
try:
    validate_columns([], ["id", "value"])
except ValueError as e:
    print("Caught error:", e)

**Expected Output:**
```
Caught error: Data is empty -- nothing to validate
```

### Try It Yourself

Test `validate_columns` with data that has EXTRA columns (more columns than required). Does it still pass? It should -- extra columns are fine.

In [ ]:
# Try it: data with extra columns
extra_data = [
    {"id": 1, "value": "10", "status": "ok", "bonus_col": "extra"},
]
# TODO: call validate_columns and see what happens


---
## Part 2: Type Validation

Checking column names is not enough. What if the `value` column contains the string `"abc"` instead of a number? Your pipeline will crash during analysis, not during validation.

Let us add type checking to catch this early.

In [ ]:
def validate_types(data, column_types, check_rows=5):
    """Check that column values match expected types.
    
    Args:
        data: list of dicts
        column_types: dict mapping column name to expected type
            Supported types: "numeric", "string", "boolean"
        check_rows: how many rows to check (default 5)
    
    Returns:
        list of error messages (empty list = all good)
    """
    errors = []
    
    for i, row in enumerate(data[:check_rows]):
        for col, expected_type in column_types.items():
            if col not in row:
                continue  # missing columns handled elsewhere
            
            val = row[col]
            
            if expected_type == "numeric":
                try:
                    float(val)
                except (ValueError, TypeError):
                    errors.append(
                        "Row " + str(i) + ": column " + repr(col)
                        + " = " + repr(val)
                        + " is not numeric"
                    )
            
            elif expected_type == "string":
                if not isinstance(val, str):
                    errors.append(
                        "Row " + str(i) + ": column " + repr(col)
                        + " = " + repr(val)
                        + " is not a string"
                    )
    
    return errors

# Test: good types
data = [
    {"id": 1, "value": "25.0", "status": "ok"},
    {"id": 2, "value": "30.0", "status": "ok"},
]
type_rules = {"value": "numeric", "status": "string"}
errors = validate_types(data, type_rules)
print("Errors:", errors)
print("Result:", "PASS" if not errors else "FAIL")

**Expected Output:**
```
Errors: []
Result: PASS
```

### Example: Catching Type Errors

In [ ]:
# Data with type problems
bad_types = [
    {"id": 1, "value": "25.0", "status": "ok"},
    {"id": 2, "value": "N/A", "status": "ok"},     # not numeric!
    {"id": 3, "value": "abc", "status": "ok"},     # not numeric!
]

errors = validate_types(bad_types, {"value": "numeric"})
print("Type errors found:")
for e in errors:
    print("  " + e)

**Expected Output:**
```
Type errors found:
  Row 1: column 'value' = 'N/A' is not numeric
  Row 2: column 'value' = 'abc' is not numeric
```

### Common Mistakes: Type Validation

**Mistake 1:** Checking `isinstance(val, int)` on CSV data -- CSV values are always strings!

**Fix:** Always try `float(val)` for numeric checks on raw CSV data.

**Mistake 2:** Only checking the first row -- the first row might be fine, but row 50 has a problem.

**Fix:** Check at least the first 5-10 rows. For small datasets, check all rows.

**Mistake 3:** Using `assert` instead of raising ValueError -- assert can be disabled with -O flag.

**Fix:** Use `raise ValueError(message)` for validation. Reserve assert for tests.


---
## Part 3: The Complete validate_schema() Function

Now let us combine column checking and type checking into one function that does everything:

In [ ]:
def validate_schema(data, config):
    """Validate that data matches the expected schema.
    
    Checks:
    1. Data is not empty
    2. All required columns are present
    3. Column types match expectations
    
    Args:
        data (list[dict]): Raw data rows
        config (dict): Must contain:
            - required_columns (list): column names
            - column_types (dict): {col: type_name}
    
    Returns:
        bool: True if valid
    
    Raises:
        ValueError: if validation fails, with a detailed message
    """
    # Check 1: not empty
    if not data:
        raise ValueError(
            "Data is empty -- cannot validate schema.\n"
            "Check that load_data() returned data."
        )
    
    # Check 2: required columns
    required = config.get("required_columns", [])
    actual = set(data[0].keys())
    missing = set(required) - actual
    
    if missing:
        extra = actual - set(required)
        msg = "Missing " + str(len(missing)) + " required column(s): " + str(sorted(missing)) + "\n"
        msg += "Available columns: " + str(sorted(actual)) + "\n"
        if extra:
            msg += "Did you mean one of: " + str(sorted(extra)[:3]) + "?"
        raise ValueError(msg)
    
    # Check 3: column types
    type_rules = config.get("column_types", {})
    errors = []
    for i, row in enumerate(data[:5]):
        for col, expected_type in type_rules.items():
            if col in row:
                val = row[col]
                if expected_type == "numeric":
                    try:
                        float(val)
                    except (ValueError, TypeError):
                        errors.append(
                            "Row " + str(i) + ": " + repr(col)
                            + " = " + repr(val)
                            + " is not numeric"
                        )
    
    if errors:
        raise ValueError(
            "Type errors found:\n" + "\n".join(errors)
        )
    
    n_cols = len(required)
    n_rows = len(data)
    print("Schema valid: " + str(n_cols) + " required columns, "
          + str(n_rows) + " rows checked")
    return True

# === Test 1: Good data ===
good_data = [
    {"id": 1, "value": "25.0", "status": "ok"},
    {"id": 2, "value": "30.0", "status": "ok"},
    {"id": 3, "value": "15.5", "status": "warning"},
]
config = {
    "required_columns": ["id", "value", "status"],
    "column_types": {"value": "numeric"},
}

result = validate_schema(good_data, config)
print("Result:", result)

**Expected Output:**
```
Schema valid: 3 required columns, 3 rows checked
Result: True
```

### Test: Missing Columns

In [ ]:
# Test: missing columns give a helpful error
bad_data = [{"id": 1, "temperature": 25.0}]

try:
    validate_schema(bad_data, config)
except ValueError as e:
    print("Error message:")
    print(e)

**Expected Output:**
```
Error message:
Missing 2 required column(s): ['status', 'value']
Available columns: ['id', 'temperature']
Did you mean one of: ['temperature']?
```

### Test: Type Errors

In [ ]:
# Test: type errors in data
type_bad = [
    {"id": 1, "value": "25.0", "status": "ok"},
    {"id": 2, "value": "not_a_number", "status": "ok"},
]

try:
    validate_schema(type_bad, config)
except ValueError as e:
    print("Type error caught:")
    print(e)

**Expected Output:**
```
Type error caught:
Type errors found:
Row 1: 'value' = 'not_a_number' is not numeric
```

### Try It Yourself

Create your own test data with 3 rows that has a DIFFERENT type error (e.g., an id column that should be numeric but contains text). Verify that validate_schema catches it.

In [ ]:
# Try it: create data with a type error
# TODO: write your test here


---
## Part 4: Meaningful Error Messages

The difference between a beginner and a professional programmer is not whether their code has errors -- it is how HELPFUL those errors are.

Compare these two approaches:

### Bad vs. Good Error Messages

```
BAD:   AssertionError
       (What failed? Where? What should I do?)

BAD:   KeyError: 'temp'
       (Which function? What keys are available?)

GOOD:  ValueError: Missing column 'temp'.
       Available columns: ['temperature', 'id', 'status'].
       Did you mean 'temperature'?
       (Clear, actionable, suggests a fix!)
```

In [ ]:
# BAD error handling -- DO NOT do this
def bad_validate(data):
    """This gives unhelpful errors."""
    assert len(data) > 0                 # AssertionError (no message)
    assert "value" in data[0]             # Which column is missing?
    assert data[0]["value"] != ""          # Why? What is wrong?

# GOOD error handling -- DO this
def good_validate(data, required_columns):
    """This gives clear, actionable errors."""
    if not data:
        raise ValueError(
            "Cannot validate empty dataset. "
            "Check that load_data() returned data."
        )
    
    actual = set(data[0].keys())
    missing = set(required_columns) - actual
    extra = actual - set(required_columns)
    
    if missing:
        msg = "Missing " + str(len(missing)) + " required column(s): " + str(sorted(missing)) + "\n"
        msg += "Available columns: " + str(sorted(actual)) + "\n"
        if extra:
            msg += "Hint: did you mean " + str(sorted(extra)[:3]) + "?"
        raise ValueError(msg)
    
    print("Validation passed!")
    return True

# Demo: the good version gives helpful errors
try:
    good_validate(
        [{"temperature": 25, "pressure": 101}],
        ["temp", "press", "humidity"]
    )
except ValueError as e:
    print("Error:")
    print(e)

**Expected Output:**
```
Error:
Missing 3 required column(s): ['humidity', 'press', 'temp']
Available columns: ['pressure', 'temperature']
Hint: did you mean ['pressure', 'temperature']?
```

### Key Takeaway

- Always raise ValueError (not assert) for validation errors
- Include WHAT is wrong, WHAT was expected, and WHAT the user should do
- Suggest possible fixes when you can (e.g., 'did you mean...')
- The error message is part of your user interface

---
## Part 5: Data Contracts

A **data contract** is a formal specification of what your pipeline expects. It lives in your config and acts as documentation AND validation rules.

Think of it like a contract between the data producer and your pipeline: "I promise to give you data that looks like THIS, and you promise to process it correctly."

In [ ]:
def create_data_contract():
    """Create a data contract for the v2 pipeline.
    
    A data contract specifies:
    - Which columns must exist
    - What types each column should have
    - What ranges are acceptable
    - Which columns cannot be empty
    """
    return {
        "required_columns": ["id", "value", "timestamp", "status"],
        "column_types": {
            "id": "numeric",
            "value": "numeric",
            "timestamp": "string",
            "status": "string",
        },
        "value_ranges": {
            "value": (0, 100),
        },
        "non_null_columns": ["id", "value"],
    }

contract = create_data_contract()
print("Data Contract:")
for key, val in contract.items():
    print("  " + str(key) + ": " + str(val))

**Expected Output:**
```
Data Contract:
  required_columns: ['id', 'value', 'timestamp', 'status']
  column_types: {'id': 'numeric', 'value': 'numeric', 'timestamp': 'string', 'status': 'string'}
  value_ranges: {'value': (0, 100)}
  non_null_columns: ['id', 'value']
```

### Using the Contract for Validation

In [ ]:
def validate_with_contract(data, contract):
    """Full validation using a data contract.
    
    Returns:
        dict with "valid" (bool), "errors" (list), "warnings" (list)
    """
    result = {"valid": True, "errors": [], "warnings": []}
    
    # Check: not empty
    if not data:
        result["valid"] = False
        result["errors"].append("Data is empty")
        return result
    
    # Check: required columns
    actual = set(data[0].keys())
    missing = set(contract.get("required_columns", [])) - actual
    if missing:
        result["valid"] = False
        result["errors"].append("Missing columns: " + str(sorted(missing)))
    
    # Check: types (first 5 rows)
    for i, row in enumerate(data[:5]):
        for col, expected in contract.get("column_types", {}).items():
            if col not in row:
                continue
            val = row[col]
            if expected == "numeric":
                try:
                    float(val)
                except (ValueError, TypeError):
                    result["errors"].append(
                        "Row " + str(i) + ": " + repr(col)
                        + " = " + repr(val) + " is not numeric"
                    )
                    result["valid"] = False
    
    # Check: non-null columns
    for col in contract.get("non_null_columns", []):
        null_count = sum(1 for row in data
                         if not row.get(col) or str(row[col]).strip() == "")
        if null_count > 0:
            pct = round(null_count / len(data) * 100, 1)
            result["warnings"].append(
                "Column " + repr(col) + " has " + str(null_count)
                + " null values (" + str(pct) + "%)"
            )
    
    return result

# Test with mixed data
test_data = [
    {"id": "1", "value": "25.0", "timestamp": "2024-01-01", "status": "ok"},
    {"id": "2", "value": "", "timestamp": "2024-01-02", "status": "ok"},
    {"id": "3", "value": "30.0", "timestamp": "2024-01-03", "status": "warn"},
]

contract = create_data_contract()
result = validate_with_contract(test_data, contract)

print("Valid:", result["valid"])
print("Errors:", result["errors"])
print("Warnings:", result["warnings"])

**Expected Output:**
```
Valid: True
Errors: []
Warnings: ["Column 'value' has 1 null values (33.3%)"]
```

---
## Part 6: Integrating Validation Into Your Pipeline

Validation should happen EARLY -- right after loading data, before any cleaning or analysis.

### Where Validation Fits

```
load_data()
    |
    v
validate_schema()  <--- NEW in v2
    |
    v
clean_data()
    |
    v
analyze()
    |
    v
plot() + export_results()
```

In [ ]:
def load_and_validate(config):
    """Load data and validate schema before processing.
    
    Returns:
        list of dicts if valid, None if validation fails
    """
    # Step 1: Load (simulated)
    data = [
        {"id": 1, "value": "25.0", "timestamp": "2024-01-01"},
        {"id": 2, "value": "30.0", "timestamp": "2024-01-02"},
        {"id": 3, "value": "", "timestamp": "2024-01-03"},
    ]
    print("Loaded " + str(len(data)) + " rows")
    
    # Step 2: Validate schema BEFORE cleaning
    try:
        validate_schema(data, config)
        print("Schema validation passed!")
    except ValueError as e:
        print("Schema validation FAILED:")
        print(str(e))
        print("Fix your data or config before continuing.")
        return None
    
    return data

config = {
    "required_columns": ["id", "value", "timestamp"],
    "column_types": {"value": "numeric"},
}
result = load_and_validate(config)
if result:
    print("Ready to process " + str(len(result)) + " rows")

**Expected Output:**
```
Loaded 3 rows
Schema valid: 3 required columns, 3 rows checked
Schema validation passed!
Ready to process 3 rows
```

### Debugging Tips: Schema Validation

- If you get 'Data is empty', check that load_data() actually read the file.
- If you get 'Missing columns', print data[0].keys() to see what columns exist.
- If column names have extra spaces, use .strip() on them before validation.
- CSV files always load values as strings -- that is why type checking uses float().

---
## Part 7: Common Mistakes With Worked Fixes

### Mistake 1: Using assert for Validation

In [ ]:
# WRONG: using assert for validation
def bad_validate_v1(data):
    assert len(data) > 0           # No message!
    assert "value" in data[0]       # Cryptic if it fails

# What happens when it fails:
try:
    bad_validate_v1([])
except AssertionError as e:
    print("AssertionError:", e)  # Empty message!

# CORRECT: use raise ValueError with a message
def good_validate_v1(data):
    if not data:
        raise ValueError(
            "Data is empty. Check that your file path is correct "
            "and the file is not empty."
        )

try:
    good_validate_v1([])
except ValueError as e:
    print("ValueError:", e)

**Expected Output:**
```
AssertionError: 
ValueError: Data is empty. Check that your file path is correct and the file is not empty.
```

### Mistake 2: Checking Types With isinstance on CSV Data

In [ ]:
# WRONG: CSV data is always strings
row = {"value": "25.0"}  # loaded from CSV
print("Is int?", isinstance(row["value"], int))       # False!
print("Is float?", isinstance(row["value"], float))   # False!
print("Is str?", isinstance(row["value"], str))       # True (always)

# CORRECT: try to convert
try:
    val = float(row["value"])
    print("Converted to float:", val)  # Works!
except ValueError:
    print("Not a number")

**Expected Output:**
```
Is int? False
Is float? False
Is str? True
Converted to float: 25.0
```

### Mistake 3: Only Checking the First Row

In [ ]:
# WRONG: first row is fine, but row 3 has a problem
tricky_data = [
    {"id": "1", "value": "25.0"},   # fine
    {"id": "2", "value": "30.0"},   # fine
    {"id": "3", "value": "N/A"},    # PROBLEM
    {"id": "4", "value": "40.0"},   # fine
]

# Only checking row 0 would miss the problem!
print("Row 0 value:", tricky_data[0]["value"])
print("Can convert?", end=" ")
try:
    float(tricky_data[0]["value"])
    print("Yes")
except ValueError:
    print("No")

# Better: check multiple rows
for i, row in enumerate(tricky_data):
    try:
        float(row["value"])
    except (ValueError, TypeError):
        print("Problem at row " + str(i) + ": value = " + repr(row["value"]))

**Expected Output:**
```
Row 0 value: 25.0
Can convert? Yes
Problem at row 2: value = 'N/A'
```

---
## Try It Yourself: Practice Exercises

### Exercise 1: Validate a Custom Schema
Write a schema config for data with columns: name (string), age (numeric), email (string). Then test it with both valid and invalid data.

In [ ]:
# Exercise 1: Your custom schema validation
# TODO: create config with required_columns and column_types
# TODO: create test_data_good and test_data_bad
# TODO: call validate_schema on both


### Exercise 2: Add Range Validation
Extend validate_schema to also check value ranges. If config has `value_ranges = {'age': (0, 150)}`, reject rows where age is outside that range.

In [ ]:
# Exercise 2: Add range checking
# TODO: write validate_ranges(data, config) function
# It should return a list of violations


### Exercise 3: Validation Report
Write a function that runs ALL validation checks and returns a summary dict.

In [ ]:
# Exercise 3: Full validation report
# TODO: write full_validation_report(data, config)
# It should return {"valid": bool, "checks_passed": int, "checks_failed": int, "details": [...]}


---
## Mini-Quiz

In [ ]:
# Q1: What is the difference between assert and raise ValueError?
# Answer: 

# Q2: Why should validation happen BEFORE cleaning, not after?
# Answer: 

# Q3: What 3 things should a good error message include?
# Answer: 

# Q4: Why does isinstance(csv_value, float) not work on CSV data?
# Answer: 

---
## Homework: 12 Exercises

Complete before next week. Estimated time: 2-4 hours.

### Review (1-4)

In [ ]:
# HW1: Explain in comments: what is a data contract?
# What are the 4 things it specifies?
# Answer:


In [ ]:
# HW2: Write a validate_schema call that checks for columns
# "name", "email", "phone" with "phone" being numeric.
# Test with data that PASSES and data that FAILS.


In [ ]:
# HW3: What error message would validate_schema give for this data?
# data = [{"x": 1, "y": 2}]
# config = {"required_columns": ["a", "b", "c"], "column_types": {}}
# Write your prediction, then run it to check.
# Prediction:


In [ ]:
# HW4: Fix this broken validation function:
def broken_validate(data, columns):
    for col in columns:
        if col in data:
            return True
    return False

# What is wrong? Fix it below:
# def fixed_validate(data, columns):


### Practice (5-8)

In [ ]:
# HW5: Write validate_non_null(data, non_null_columns) that checks
# that certain columns never have empty string or None values.
# Return a dict of {column: null_count} for any column with nulls.


In [ ]:
# HW6: Write validate_unique(data, unique_columns) that checks
# that certain columns have no duplicate values.
# Return a dict of {column: duplicate_count} for any column with duplicates.


In [ ]:
# HW7: Create a data contract for YOUR track project.
# Include: required_columns, column_types, value_ranges, non_null_columns.
# Test it with sample data from your project.


In [ ]:
# HW8: Write a function that validates a CSV file from disk:
# def validate_csv_file(filepath, config):
#     1. Check file exists
#     2. Load with csv.DictReader
#     3. Call validate_schema
#     4. Return validation result


### Challenge (9-11)

In [ ]:
# HW9: Implement "fuzzy column matching" -- if a required column is
# missing but a similar column exists, suggest it.
# Example: required="temp", actual has "temperature" -> suggest it
# Hint: check if required_col is a substring of any actual column.


In [ ]:
# HW10: Write a validation decorator that automatically validates
# the input to any pipeline function:
# @validate_input(required_columns=["id", "value"])
# def clean_data(data, config):
#     ...


In [ ]:
# HW11: Create a ValidationReport class that collects errors,
# warnings, and info messages, and can print a formatted summary.


### Mini-Project

In [ ]:
# HW12 (Mini-Project): Build a complete schema validation module.
# Requirements:
# - validate_schema(data, config) function
# - At least 5 different checks (columns, types, ranges, nulls, uniqueness)
# - Clear error messages with suggestions
# - A demo that shows it catching 3 different kinds of errors
# - Print a validation summary at the end


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)